In [1]:
# =========================================================================
# 📘 TRAIN_RANDOM_FOREST.PY - VERSIÓN CORREGIDA
# =========================================================================
# ✅ CORRECCIÓN: Manejo robusto de labels y class_names

import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [2]:
# =========================================================================
# ⚙️ CONFIGURACIÓN
# =========================================================================
class Config:
    EMBEDDINGS_PATH = "../backend/models/embeddings.pkl"
    MODEL_OUTPUT_PATH = "../backend/models"
    MODEL_FILENAME = "random_forest_model.pkl"
    
    # Hiperparámetros Random Forest
    N_ESTIMATORS = 100
    MAX_DEPTH = 20
    MIN_SAMPLES_SPLIT = 5
    MIN_SAMPLES_LEAF = 2
    RANDOM_STATE = 42
    
    # Train/Test split
    TEST_SIZE = 0.2
    
    # Visualización
    GENERATE_PLOTS = True

print("=" * 70)
print("🌲 ENTRENAMIENTO DE RANDOM FOREST")
print("=" * 70)

🌲 ENTRENAMIENTO DE RANDOM FOREST


In [3]:
# =========================================================================
# PASO 1: CARGAR DATOS
# =========================================================================
print("\n📂 PASO 1: Cargando embeddings...")

with open(Config.EMBEDDINGS_PATH, 'rb') as f:
    data = pickle.load(f)

# Extraer datos
X = data['embeddings'].astype(np.float32)
y = data['labels']
class_names = data['class_names']

# ✅ NUEVO: Cargar mapeo de labels (si existe)
label_to_class = data.get('label_to_class', None)

print(f"✅ Datos cargados correctamente")
print(f"   📊 Features (X): {X.shape}")
print(f"   🏷️  Labels (y): {y.shape}")
print(f"   📋 Clases: {len(class_names)}")
print(f"   🔧 Modelo embeddings: {data['model']}")
print(f"   📏 Dimensiones: {data['embedding_dim']}")



📂 PASO 1: Cargando embeddings...
✅ Datos cargados correctamente
   📊 Features (X): (20760, 1280)
   🏷️  Labels (y): (20760,)
   📋 Clases: 21
   🔧 Modelo embeddings: MobileNetV2
   📏 Dimensiones: 1280


In [4]:
# =========================================================================
# PASO 2: ANÁLISIS EXPLORATORIO
# =========================================================================
print("\n📊 PASO 2: Análisis exploratorio de datos")

print(f"\n🔍 Información del dataset:")
print(f"   Total de muestras: {len(X):,}")
print(f"   Features por muestra: {X.shape[1]}")
print(f"   Rango de labels: [{y.min()}, {y.max()}]")
print(f"   Labels únicos: {len(np.unique(y))}")

# Estadísticas de embeddings
print(f"\n📈 Estadísticas de embeddings:")
print(f"   Media: {X.mean():.4f}")
print(f"   Desv. estándar: {X.std():.4f}")
print(f"   Mínimo: {X.min():.4f}")
print(f"   Máximo: {X.max():.4f}")


📊 PASO 2: Análisis exploratorio de datos

🔍 Información del dataset:
   Total de muestras: 20,760
   Features por muestra: 1280
   Rango de labels: [0, 20]
   Labels únicos: 21

📈 Estadísticas de embeddings:
   Media: 0.4674
   Desv. estándar: 0.6554
   Mínimo: 0.0000
   Máximo: 6.0000


In [5]:
# =========================================================================
# PASO 3: ✅ DISTRIBUCIÓN DE CLASES (CORREGIDO)
# =========================================================================
print("\n📊 PASO 3: Distribución de clases")

unique_labels, counts = np.unique(y, return_counts=True)

# ✅ MAPEO ROBUSTO: Usar label_to_class si existe, sino crear dinámicamente
if label_to_class is not None:
    # Usar el mapeo guardado
    class_names_mapped = [label_to_class.get(label, f"clase_{label}") 
                          for label in unique_labels]
else:
    # Crear mapeo dinámico si no existe
    print("⚠️ Advertencia: No se encontró 'label_to_class'. Creando mapeo automático...")
    
    # Verificar si los labels son consecutivos desde 0
    if np.array_equal(unique_labels, np.arange(len(unique_labels))):
        # Labels consecutivos: usar class_names directamente
        class_names_mapped = [class_names[i] if i < len(class_names) else f"clase_{i}" 
                              for i in unique_labels]
    else:
        # Labels no consecutivos: crear nombres genéricos
        label_to_class = {label: f"clase_{label}" for label in unique_labels}
        class_names_mapped = [label_to_class[label] for label in unique_labels]
        print(f"   Creados {len(label_to_class)} nombres de clase automáticamente")

# Crear DataFrame de distribución
class_distribution = pd.DataFrame({
    'Label': unique_labels,
    'Clase': class_names_mapped,
    'Cantidad': counts,
    'Porcentaje': (counts / len(y) * 100).round(2)
})

print("\n📊 Distribución de clases:")
print(class_distribution.to_string(index=False))

# Verificar balance de clases
min_samples = counts.min()
max_samples = counts.max()
balance_ratio = min_samples / max_samples

print(f"\n⚖️ Balance de clases:")
print(f"   Clase con menos muestras: {min_samples}")
print(f"   Clase con más muestras: {max_samples}")
print(f"   Ratio de balance: {balance_ratio:.2%}")

if balance_ratio < 0.5:
    print("   ⚠️ Dataset desbalanceado. Considera usar class_weight='balanced'")
else:
    print("   ✅ Dataset relativamente balanceado")


📊 PASO 3: Distribución de clases

📊 Distribución de clases:
 Label             Clase  Cantidad  Porcentaje
     0          pancakes       978        4.71
     1           waffles       977        4.71
     2      french_toast       976        4.70
     3     eggs_benedict       979        4.72
     4    scrambled_eggs       978        4.71
     5          omelette       978        4.71
     6        fried_eggs       978        4.71
     7             bacon      1000        4.82
     8           sausage       979        4.72
     9       hash_browns       979        4.72
    10             toast       979        4.72
    11             bagel       979        4.72
    12         croissant      1000        4.82
    13            muffin      1000        4.82
    14            cereal      1000        4.82
    15           oatmeal      1000        4.82
    16            yogurt      1000        4.82
    17       fruit_salad      1000        4.82
    18     smoothie_bowl      1000        4.82

In [6]:
#=========================================================================
# PASO 4: DIVISIÓN TRAIN/TEST
# =========================================================================
print(f"\n🔀 PASO 4: División Train/Test ({int((1-Config.TEST_SIZE)*100)}%/{int(Config.TEST_SIZE*100)}%)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=Config.TEST_SIZE,
    random_state=Config.RANDOM_STATE,
    stratify=y  # Mantener la proporción de clases
)

print(f"✅ División completada:")
print(f"   Train: {X_train.shape[0]:,} muestras ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Test:  {X_test.shape[0]:,} muestras ({len(X_test)/len(X)*100:.1f}%)")


🔀 PASO 4: División Train/Test (80%/20%)
✅ División completada:
   Train: 16,608 muestras (80.0%)
   Test:  4,152 muestras (20.0%)


In [7]:
# =========================================================================
# PASO 5: ENTRENAR RANDOM FOREST
# =========================================================================
print(f"\n🌲 PASO 5: Entrenando Random Forest...")
print(f"   Árboles: {Config.N_ESTIMATORS}")
print(f"   Profundidad máxima: {Config.MAX_DEPTH}")
print(f"   Min samples split: {Config.MIN_SAMPLES_SPLIT}")
print(f"   Min samples leaf: {Config.MIN_SAMPLES_LEAF}")

rf_model = RandomForestClassifier(
    n_estimators=Config.N_ESTIMATORS,
    max_depth=Config.MAX_DEPTH,
    min_samples_split=Config.MIN_SAMPLES_SPLIT,
    min_samples_leaf=Config.MIN_SAMPLES_LEAF,
    random_state=Config.RANDOM_STATE,
    n_jobs=-1,  # Usar todos los cores disponibles
    verbose=1
)

rf_model.fit(X_train, y_train)

print("\n✅ Entrenamiento completado!")



🌲 PASO 5: Entrenando Random Forest...
   Árboles: 100
   Profundidad máxima: 20
   Min samples split: 5
   Min samples leaf: 2


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    3.2s



✅ Entrenamiento completado!


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    9.8s finished


In [8]:
# =========================================================================
# PASO 6: EVALUACIÓN
# =========================================================================
print("\n📈 PASO 6: Evaluación del modelo")

# Predicciones
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"\n🎯 Accuracy:")
print(f"   Train: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"   Test:  {test_acc:.4f} ({test_acc*100:.2f}%)")

# Diferencia (overfitting?)
diff = train_acc - test_acc
if diff > 0.05:
    print(f"   ⚠️ Posible overfitting (diferencia: {diff:.4f})")
else:
    print(f"   ✅ Buen balance train/test")

# Reporte de clasificación
print("\n📋 Reporte de clasificación (Test):")
print(classification_report(
    y_test, 
    y_pred_test,
    target_names=class_names_mapped,
    digits=4
))

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s



📈 PASO 6: Evaluación del modelo

🎯 Accuracy:
   Train: 0.9999 (99.99%)
   Test:  0.4822 (48.22%)
   ⚠️ Posible overfitting (diferencia: 0.5178)

📋 Reporte de clasificación (Test):
                   precision    recall  f1-score   support

         pancakes     0.3014    0.1122    0.1636       196
          waffles     0.6174    0.7282    0.6682       195
     french_toast     0.3410    0.3026    0.3207       195
    eggs_benedict     0.4583    0.3929    0.4231       196
   scrambled_eggs     0.5112    0.4667    0.4879       195
         omelette     0.5116    0.4490    0.4783       196
       fried_eggs     0.3333    0.2769    0.3025       195
            bacon     0.5023    0.5350    0.5182       200
          sausage     0.5992    0.7245    0.6559       196
      hash_browns     0.5090    0.7194    0.5962       196
            toast     0.4968    0.3980    0.4419       196
            bagel     0.6327    0.7908    0.7029       196
        croissant     0.6139    0.6200    0.6169   

[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.3s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [9]:
# =========================================================================
# PASO 7: GUARDAR MODELO
# =========================================================================
print("\n💾 PASO 7: Guardando modelo...")

output_path = Path(Config.MODEL_OUTPUT_PATH)
output_path.mkdir(parents=True, exist_ok=True)

model_filepath = output_path / Config.MODEL_FILENAME

# ✅ GUARDAR CON MAPEO DE LABELS
model_data = {
    'model': rf_model,
    'class_names': class_names,
    'label_to_class': label_to_class if label_to_class else {i: name for i, name in enumerate(class_names)},
    'accuracy': test_acc,
    'n_classes': len(unique_labels),
    'embedding_dim': X.shape[1],
    'model_type': 'RandomForest'
}

with open(model_filepath, 'wb') as f:
    pickle.dump(model_data, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"✅ Modelo guardado en: {model_filepath}")



💾 PASO 7: Guardando modelo...
✅ Modelo guardado en: ..\backend\models\random_forest_model.pkl


In [10]:
# =========================================================================
# PASO 8: VISUALIZACIONES (OPCIONAL)
# =========================================================================
if Config.GENERATE_PLOTS:
    print("\n📊 PASO 8: Generando visualizaciones...")
    
    # 1. Matriz de confusión
    cm = confusion_matrix(y_test, y_pred_test)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names_mapped,
        yticklabels=class_names_mapped
    )
    plt.title('Matriz de Confusión', fontsize=14, fontweight='bold')
    plt.xlabel('Predicción', fontsize=12)
    plt.ylabel('Real', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    cm_path = output_path / 'confusion_matrix.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"   ✅ Matriz de confusión: {cm_path}")
    
    # 2. Importancia de features (top 20)
    feature_importance = rf_model.feature_importances_
    top_indices = np.argsort(feature_importance)[-20:][::-1]
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(top_indices)), feature_importance[top_indices], color='forestgreen')
    plt.xlabel('Importancia', fontsize=12, fontweight='bold')
    plt.ylabel('Feature Index', fontsize=12, fontweight='bold')
    plt.title('Top 20 Features más importantes', fontsize=14, fontweight='bold')
    plt.yticks(range(len(top_indices)), top_indices)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    fi_path = output_path / 'feature_importance.png'
    plt.savefig(fi_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"   ✅ Importancia de features: {fi_path}")


📊 PASO 8: Generando visualizaciones...
   ✅ Matriz de confusión: ..\backend\models\confusion_matrix.png
   ✅ Importancia de features: ..\backend\models\feature_importance.png


In [11]:
# =========================================================================
# RESUMEN FINAL
# =========================================================================
print("\n" + "="*70)
print("✨ ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
print("="*70)
print(f"\n📊 Resumen:")
print(f"   🎯 Accuracy Test: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   📁 Modelo guardado: {model_filepath}")
print(f"   📋 Clases: {len(unique_labels)}")
print(f"   🌲 Árboles: {Config.N_ESTIMATORS}")
print("\n💡 Próximos pasos:")
print("   1. Evaluar el modelo con nuevas imágenes")
print("   2. Ajustar hiperparámetros si es necesario")
print("   3. Implementar en producción")
print("="*70)


✨ ENTRENAMIENTO COMPLETADO EXITOSAMENTE

📊 Resumen:
   🎯 Accuracy Test: 0.4822 (48.22%)
   📁 Modelo guardado: ..\backend\models\random_forest_model.pkl
   📋 Clases: 21
   🌲 Árboles: 100

💡 Próximos pasos:
   1. Evaluar el modelo con nuevas imágenes
   2. Ajustar hiperparámetros si es necesario
   3. Implementar en producción
